## Assigmnet 1 - Erling

Link to github: https://github.com/Erlinghm/Erlinghm-ind320.git

Link to video: https://youtu.be/4t8LMsxiQRw

Link to streamlit: https://erlinghm-ind320.streamlit.app/


### Imports

In [ ]:
# Import pandas for data handling, requests for the API call, and matplotlib for plotting
import pandas as pd
import requests
import matplotlib.pyplot as plt

### Read the magasinstatistikk data from the NVE API

For CA1, we were given the data directly as a local reservoirs.csv file - no API call was involved at that point. Afterwards, I replaced that static file with a live call to NVE's Magasinstatistikk API instead, so the notebook (and the Streamlit app) always reflects the newest published week rather than a frozen snapshot from when the file was handed out.

In [ ]:
# Fetch the full magasinstatistikk history directly from NVE's API (no parameters needed - it returns everything)
MAGASIN_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"
response = requests.get(MAGASIN_URL, timeout=30)
response.raise_for_status()  # Raise an error early if the API call failed

df = pd.DataFrame(response.json())
df["dato_Id"] = pd.to_datetime(df["dato_Id"])  # API returns clean ISO date strings, so this just works

print(df.shape)
df.head()  # Preview

### Rename headers to English and understandable

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "unique_values": df.nunique()
})
summary # Print a summary over the data types and the number of unique values in each column

In [ ]:
df = df.rename(columns={
    "omrType": "area_type",                               # 3 typs of area, "EL", "VASS" AND "NO"
    "omrnr": "area_number",                               # EL is split into 5 different electrical pricing zones,
    # VASS is split into 3 differnt draining regions and  NO is the hole conttry under 1
    "iso_aar": "iso_year",                                # data year in ISO format
    "iso_uke": "iso_week",                                # data week in ISO format
    "fyllingsgrad": "fill_degree",                        # reservoir fill degree (0-1 fraction)
    "kapasitet_TWh": "capacity_twh",                      # total capacity in TWh
    "fylling_TWh": "filling_twh",                         # actual stored energy in TWh
    "neste_Publiseringsdato": "next_publish_date",        # next publication date
    "fyllingsgrad_forrige_uke": "fill_degree_prev_week",  # fill degree from previous week
    "endring_fyllingsgrad": "fill_degree_change",         # change in fill degree
})
df.head()

### Print its contents in a relevant way

As it is now, each week contains 9 datapoints, representing the sum of all
reservoirs within each price zone, the sum of all reservoirs within each
drainage/watercourse region, and a sum for Norway as a whole.

We will therefore choose to filter out anything that is not
area_type == "NO", meaning we will focus on Norway as a whole.

In [ ]:

national = df[df["area_type"] == "NO"].copy()

# Sort by date and use the date as the index, since this is time series data
national = national.sort_values("dato_Id").set_index("dato_Id")
national.head()

In [ ]:
# Print the shape (rows, columns) and summary information about the dataset
print(national.shape)
national.info()        # Data types and non-null counts for each column
national.describe()    # Summary statistics (mean, std, min, max, etc.)

### Plot each column separately.

In [ ]:
# The numeric columns we want to visualize
numeric_cols = ["fill_degree", "capacity_twh", "filling_twh",
                 "fill_degree_prev_week", "fill_degree_change"]

# Plot each column in its own separate figure
for col in numeric_cols:
    plt.figure(figsize=(10, 3))
    plt.plot(national.index, national[col])
    plt.title(col)
    plt.xlabel("Date")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

### Plot all columns together

When plotting each feature toghtere, we have to deal with the features being in different scales. We therefore chosce to normalize the data, giving it a range between 0 and 1, using min mag scaling. 

In [ ]:
# Normalize the columns to a 0-1 range
normalized = (national[numeric_cols] - national[numeric_cols].min()) / (national[numeric_cols].max() - national[numeric_cols].min())

plt.figure(figsize=(10, 5))
for col in numeric_cols:
    plt.plot(normalized.index, normalized[col], label=col)
plt.title("Reservoir data over time (national) - normalized to 0-1")
plt.xlabel("Date")
plt.ylabel("Normalized value")
plt.legend()
plt.tight_layout()
plt.show()

### Comment on plot

From the plot we find that fill_degree, capacity_twh, and filling_twh
are following each other pretty closely, which makes sense because they are
basically representing the same underlying information. fill_degree_prev_week
is lagging a bit behind, of course, since it is simply the previous data point
repeated. fill_degree_change, on the other hand, is fairly uncorrelated with
the other features since it is based on the change rather than the level.

### Work log

I started this assignment by getting familiar with the dataset we were given. For this reason, the notebook was really helpful. I read in the dataset with pandas and spent some time understanding how it was structured, especially the relationship between area_type and area_number. I discussed this with AI as well.

For displaying the content in a relevant way, I'm sure there are multiple different solutions. But the clearest one I saw at the time was looking at Norway as a whole. That way, it's easy to see how the different columns change over years and seasons. I consider this a simpler kind of solution, while noting that more complex solutions are also possible.

After finishing the notebook, I moved on to setting up my Streamlit app. This was well described in the assignment, and the process of having Streamlit and GitHub work together was smooth and user-friendly for someone new to it. I found the Streamlit package quite intuitive and easy to work with. In data_utils.py, I made sure that the column-renaming step only runs once when the app starts up, not every time something happens inside the Streamlit app.

### AI Usage

During this assignment, I used AI as a discussion partner and for help when I got stuck.

Among other things, I used it for help with GitHub, since that's something I wasn't entirely confident with. General stuff, like how to clone a repository, create a branch, and push changes, I've had help with. I also used AI as a discussion partner for things like .gitignore. I couldn't quite remember what it was, but discussed it with AI and concluded I should include one.

I also had help from AI setting up a checklist for how to set up and deploy a Streamlit app, which was very helpful. For the code, I always try to do most of it myself, but had to use AI a few times - especially for the plotting logic in the Streamlit app.